# Lab M15 — Saving and Loading Models for Automation (Solution)

> **Goal:** Save a trained model, reload it, generate predictions on new data, and document the workflow for reproducibility.  
> **Dataset:** `validated_dataset_m15.csv`

---

## How to use this notebook
- Run cells top to bottom.
- Read the **markdown instructions** carefully.
- This is the **solution** notebook with completed code and explanations.

✅ By the end, you will have:
- A saved model file (`models/revenue_model.joblib`)  
- Model metadata documenting features and metrics  
- A standalone prediction script (`scripts/predict.py`)  
- Predictions on new data saved to `outputs/predictions.csv`

## 📚 Connection to Module 14

In Module 14, you learned to **train and evaluate baseline models** using Scikit-Learn.

In this module, you'll learn to **deploy models** for real-world use:

| Module 14 (Training) | Module 15 (Deployment) |
|---------------------|------------------------|
| Train models in notebook | Save models to files |
| Evaluate on test data | Load and predict on new data |
| Manual predictions | Automated prediction scripts |
| Results in notebook | Documented, reproducible workflow |

**Key insight:** A model in a notebook cannot deliver value — deployment makes it useful.

In [ ]:
# 1) Setup
# Import all required libraries

from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Scikit-Learn imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline

# Create output folders (safe to re-run)
Path("models").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("scripts").mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print(f"📅 Current date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 2) Load the Dataset

We'll use the same validated dataset from Module 14.

In [ ]:
# Load dataset (project-root anchored so it works from /notebooks)
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "validated_dataset_m15.csv"

df = pd.read_csv(data_path)

print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head()


## 3) Train a Model (Quick Review from M14)

✅ **Solution:** We use the same feature set and approach from Module 14.

In [ ]:
# Define features and target
feature_cols = ['price_usd', 'discount_pct', 'marketing_spend_usd', 'web_sessions', 'units_sold']
target_col = 'revenue_usd'

X = df[feature_cols].copy()
y = df[target_col].copy()

print(f"✅ Features: {feature_cols}")
print(f"✅ Target: {target_col}")
print(f"✅ X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print(f"📊 Training set: {X_train.shape[0]} samples")
print(f"📊 Test set: {X_test.shape[0]} samples")

In [ ]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

print("✅ Model trained!")
print(f"\n📊 Model coefficients:")
for feature, coef in zip(feature_cols, model.coef_):
    print(f"   {feature}: {coef:.4f}")
print(f"   Intercept: {model.intercept_:.4f}")

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("📊 Model Performance on Test Set:")
print(f"   MAE:  ${mae:,.2f}")
print(f"   RMSE: ${rmse:,.2f}")

## 4) Save the Model with Joblib

✅ **Solution:** Use `joblib.dump()` to serialize the model to disk.

In [ ]:
# Save the trained model using Joblib
model_path = Path("models/revenue_model.joblib")

joblib.dump(model, model_path)

# Verify the file was created
if model_path.exists():
    file_size = model_path.stat().st_size / 1024  # Size in KB
    print(f"✅ Model saved to: {model_path}")
    print(f"📁 File size: {file_size:.2f} KB")
else:
    print("❌ Error: Model file was not created!")

## 5) Load the Model and Verify

✅ **Solution:** The loaded model produces identical predictions to the original.

In [ ]:
# Load the saved model
loaded_model = joblib.load(model_path)

print(f"✅ Model loaded from: {model_path}")
print(f"📊 Model type: {type(loaded_model).__name__}")

In [ ]:
# Verify predictions match the original model
y_pred_loaded = loaded_model.predict(X_test)

# Compare predictions
predictions_match = np.allclose(y_pred, y_pred_loaded)

if predictions_match:
    print("✅ SUCCESS: Loaded model produces identical predictions!")
else:
    print("❌ WARNING: Predictions do not match!")

# Show sample predictions
print("\n📊 Sample predictions comparison:")
comparison_df = pd.DataFrame({
    'Original': y_pred[:5],
    'Loaded': y_pred_loaded[:5],
    'Match': np.isclose(y_pred[:5], y_pred_loaded[:5])
})
display(comparison_df)

## 6) Save Model Metadata

✅ **Solution:** Metadata ensures the model can be used correctly in the future.

In [ ]:
# Create model metadata dictionary
model_metadata = {
    "model_name": "Revenue Prediction Model",
    "model_type": "LinearRegression",
    "model_file": "revenue_model.joblib",
    "version": "1.0",
    
    "training_info": {
        "training_date": datetime.now().isoformat(),
        "training_samples": len(X_train),
        "test_samples": len(X_test),
        "random_state": 42
    },
    
    "features": {
        "feature_names": feature_cols,
        "target_name": target_col,
        "n_features": len(feature_cols)
    },
    
    "performance": {
        "mae": round(mae, 2),
        "rmse": round(rmse, 2)
    },
    
    "limitations": [
        "Trained on limited data (366 records)",
        "May underperform on luxury products (>$1000 revenue)",
        "Does not account for seasonal patterns"
    ],
    
    "author": "Data Analytics Team",
    "sklearn_version": "1.3.0"
}

print("📋 Model Metadata:")
print(json.dumps(model_metadata, indent=2))

In [ ]:
# Save metadata to JSON file
metadata_path = Path("models/model_metadata.json")

with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"✅ Metadata saved to: {metadata_path}")

## 7) Create a Complete Model Package

✅ **Solution:** Combining model and metadata in one file simplifies deployment.

In [ ]:
# Create a complete model package
model_package = {
    'model': model,
    'feature_names': feature_cols,
    'target_name': target_col,
    'training_date': datetime.now().isoformat(),
    'metrics': {
        'mae': mae,
        'rmse': rmse
    },
    'version': '1.0'
}

# Save the complete package
package_path = Path("models/revenue_model_package.joblib")
joblib.dump(model_package, package_path)

print(f"✅ Model package saved to: {package_path}")
print(f"\n📦 Package contents:")
for key in model_package.keys():
    print(f"   - {key}")

## 8) Predict on New Data

✅ **Solution:** Load the package, validate features, generate predictions.

In [ ]:
# Create sample "new" data for prediction
new_data = pd.DataFrame({
    'price_usd': [29.99, 49.99, 79.00, 99.99, 149.99],
    'discount_pct': [0.05, 0.10, 0.15, 0.20, 0.25],
    'marketing_spend_usd': [50, 100, 150, 200, 250],
    'web_sessions': [500, 750, 1000, 1250, 1500],
    'units_sold': [10, 15, 20, 25, 30]
})

print("📊 New data for prediction:")
display(new_data)

In [ ]:
# Load the model package and generate predictions

# Load the complete package
loaded_package = joblib.load(package_path)

# Extract model and feature names
model_from_package = loaded_package['model']
features_from_package = loaded_package['feature_names']

print(f"📦 Loaded model package (v{loaded_package['version']})")
print(f"📅 Trained on: {loaded_package['training_date'][:10]}")
print(f"📊 Expected features: {features_from_package}")

# Ensure new data has correct features
X_new = new_data[features_from_package]

# Generate predictions
new_predictions = model_from_package.predict(X_new)

# Add predictions to dataframe
new_data_with_predictions = new_data.copy()
new_data_with_predictions['predicted_revenue'] = new_predictions
new_data_with_predictions['prediction_date'] = datetime.now().strftime('%Y-%m-%d')

print("\n🔮 Predictions:")
display(new_data_with_predictions)

In [ ]:
# Save predictions to CSV
predictions_path = Path("outputs/predictions.csv")
new_data_with_predictions.to_csv(predictions_path, index=False)

print(f"✅ Predictions saved to: {predictions_path}")

## 9) Visualize Predictions

In [ ]:
# Create visualization
plt.figure(figsize=(10, 5))

products = [f"Product {i+1}\n(${p:.2f})" for i, p in enumerate(new_data['price_usd'])]

plt.bar(products, new_predictions, color='steelblue', edgecolor='black')
plt.xlabel('Product (Price)')
plt.ylabel('Predicted Revenue (USD)')
plt.title('Revenue Predictions for New Products')

# Add value labels on bars
for i, (prod, pred) in enumerate(zip(products, new_predictions)):
    plt.text(i, pred + 50, f'${pred:,.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/figures/predictions_chart.png', dpi=150)
plt.show()

print("✅ Saved: outputs/figures/predictions_chart.png")

## 10) Create a Prediction Script

In [ ]:
# Create a standalone prediction script
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

script_path = OUT_DIR / "predict.py"

script_content = '''"""
predict.py - Standalone prediction script

Usage:
    python predict.py <input_csv> <output_csv>

Example:
    python predict.py data/new_products.csv output/predictions.csv
"""

import sys
import pandas as pd
import joblib

if len(sys.argv) != 3:
    print("Usage: python predict.py <input_csv> <output_csv>")
    sys.exit(1)

input_csv = sys.argv[1]
output_csv = sys.argv[2]

model = joblib.load("output/trained_model.joblib")
df = pd.read_csv(input_csv)

preds = model.predict(df)
df["prediction"] = preds

df.to_csv(output_csv, index=False)
print(f"Saved predictions to {output_csv}")
'''

script_path.write_text(script_content)
print(f"Created script: {script_path}")


## 11) Create a Model Card

In [ ]:
# Create Model Card content
model_card = f"""
# Model Card: Revenue Prediction Model

## Model Details
- **Model Type:** Linear Regression
- **Framework:** Scikit-Learn
- **Version:** 1.0
- **Training Date:** {datetime.now().strftime('%Y-%m-%d')}
- **Author:** Data Analytics Team

## Intended Use
- **Primary Use:** Predict product revenue based on pricing and marketing features
- **Users:** Business analysts, marketing team
- **Out-of-Scope:** Not suitable for luxury products (>$1000 base price)

## Training Data
- **Source:** validated_dataset_m15.csv
- **Size:** {len(df)} total records
- **Training Set:** {len(X_train)} records (80%)
- **Test Set:** {len(X_test)} records (20%)

## Features
| Feature | Description |
|---------|-------------|
| price_usd | Product price in USD |
| discount_pct | Discount percentage (0-1) |
| marketing_spend_usd | Marketing investment in USD |
| web_sessions | Number of website sessions |
| units_sold | Number of units sold |

## Performance Metrics
- **MAE:** ${mae:,.2f}
- **RMSE:** ${rmse:,.2f}

## Limitations
- Trained on limited data (single year)
- May underperform on seasonal patterns
- Does not account for competitor pricing
- Linear model may miss non-linear relationships

## Ethical Considerations
- Model should not be used for automated pricing decisions without human review
- Predictions should be interpreted as estimates, not guarantees

## Version History
| Version | Date | Changes |
|---------|------|---------|
| 1.0 | {datetime.now().strftime('%Y-%m-%d')} | Initial release |
"""

# Save Model Card
model_card_path = Path("models/MODEL_CARD.md")
with open(model_card_path, 'w') as f:
    f.write(model_card)

print(f"✅ Model Card saved to: {model_card_path}")

## 12) Lab Summary - Verify Deliverables

In [ ]:
# Verify all deliverables
print("📋 Lab Deliverables Checklist:")
print("="*50)

deliverables = [
    ("models/revenue_model.joblib", "Saved model file"),
    ("models/revenue_model_package.joblib", "Model package with metadata"),
    ("models/model_metadata.json", "Metadata JSON file"),
    ("models/MODEL_CARD.md", "Model Card documentation"),
    ("scripts/predict.py", "Prediction script"),
    ("outputs/predictions.csv", "Predictions on new data"),
    ("outputs/figures/predictions_chart.png", "Predictions visualization")
]

all_complete = True
for path, description in deliverables:
    exists = Path(path).exists()
    status = "✅" if exists else "❌"
    print(f"{status} {description}: {path}")
    if not exists:
        all_complete = False

print("="*50)
if all_complete:
    print("\n🎉 All deliverables complete! Ready for submission.")
else:
    print("\n⚠️ Some deliverables are missing. Please review.")

## 13) Solution Interpretation

✅ **Solution answers:**

### Solution Interpretation

- **Why saving models matters:**  
  Saving models enables reuse without retraining, which saves time and computational resources. It also enables deployment to production systems, sharing with team members, and versioning to track model changes over time.

- **Essential metadata:**  
  Feature names (so the model receives correct inputs), training date (for versioning), performance metrics (MAE, RMSE), limitations (for responsible use), and version number (for tracking changes). Without this metadata, users cannot correctly use the model.

- **Modifications for complex projects:**  
  For more complex projects, I would add: (1) automated testing to verify model performance, (2) input validation to check data types and ranges, (3) logging for debugging, (4) error handling for production robustness, and (5) model monitoring to detect performance degradation.

- **Application to Capstone:**  
  In my Capstone Project, I will save my final model with complete metadata, create a Model Card documenting intended use and limitations, and include a prediction script for reproducibility. This will demonstrate professional ML workflow practices.

## AI Reflection

✅ **Solution reflection:**

### Solution AI Reflection

**Key considerations for production deployment:**

1. **Environment consistency** — Production should match development (same library versions)
2. **Input validation** — Check data types, ranges, and missing values before prediction
3. **Error handling** — Gracefully handle unexpected inputs without crashing
4. **Monitoring** — Track model performance over time to detect degradation
5. **Scalability** — Consider batch processing vs. real-time predictions
6. **Security** — Protect model files and validate API inputs

**Connection to this lab:**
- We saved metadata to ensure environment consistency
- We created validation in our prediction script
- We documented limitations in the Model Card

**What I would do differently:**
- Add comprehensive logging for production debugging
- Implement automated tests for the prediction pipeline
- Set up model monitoring to track prediction distributions

## ✅ Lab Summary

In this lab, you:

1. **Trained a model** (quick review from M14)
2. **Saved the model** using Joblib
3. **Loaded and verified** the model works correctly
4. **Created metadata** documenting features and metrics
5. **Built a model package** combining model and metadata
6. **Generated predictions** on new data
7. **Created a prediction script** for automation
8. **Wrote a Model Card** for professional documentation

**Key takeaways:**
- Saving models enables reuse without retraining
- Metadata is essential for correct model usage
- Scripts enable automation beyond Jupyter
- Documentation makes your work professional and reproducible

**Next step:** Apply these skills in your **Capstone Project (Module 16)**!